In [1]:
import torch
from torch.utils.data import DataLoader

from ophir.register import fetch_base_trainer, fetch_finetune_trainer, load_base_model_ckpt, load_fintuned_ckpt

# from ophir.ticker import FineTuningSampler, construct_minute_agg_dataset, DayAgg, MinuteAgg
from ophir.ticker import StockHandlerDataset, StockHanlder, StockStreamerDataset, get_sp_500_symbols, get_splits
from ophir.training_models import LightningOHLCPredictor

torch.set_float32_matmul_precision("high")
# torch._functorch.config.donated_buffer = False

In [2]:
base_path = "/home/kalen/ophir/src/ophir/.ophir/data/days/stocks"
elements_per_sample = 365
response_size = 90
offset = 1
sp_500 = get_sp_500_symbols()
splits = None#get_splits()

train_stock_handler = StockHanlder(
    seq_len=elements_per_sample,
    base_path=base_path,
    stock_splits=splits,
    return_stock_id=False,
    return_streamer=True,
    offset=offset,
    max_year=2024,
    min_volume=1000,
    # winsorize_returns=True,
    shuffle=True,
)
# train_stock_handler.keep_stocks(sp_500)
# train_streamers = [train_stock_handler[stock] for stock in train_stock_handler.stocks]
# train_streamers = [streamer for streamer in train_streamers if streamer.size > 0]
train_dataset = StockHandlerDataset(
    train_stock_handler,
    response_size=response_size,
    cache_size=len(train_stock_handler.stocks),
)
train_dataloader = DataLoader(train_dataset, batch_size=64, pin_memory=True, num_workers=6)

val_stock_handler = StockHanlder(
    seq_len=elements_per_sample,
    base_path=base_path,
    stock_splits=splits,
    return_stock_id=False,
    return_streamer=True,
    offset=offset,
    min_year=2024,
    min_volume=1000,
    # winsorize_returns=True,
    shuffle=True,
)
val_stock_handler.keep_stocks(sp_500)
val_streamers = [val_stock_handler[stock] for stock in val_stock_handler.stocks]
val_streamers = [streamer for streamer in val_streamers if streamer.size > 0]
val_dataset = StockStreamerDataset(val_streamers, response_size=response_size)
val_dataloader = DataLoader(val_dataset, batch_size=64, pin_memory=True, num_workers=2, prefetch_factor=10)
# for item in dataloader:
#     break
# item
# model, ckpt_path = load_base_model_ckpt(return_ckpt_path=True)
model = LightningOHLCPredictor(512, 8, 8)
trainer = fetch_base_trainer()
# model = model.cuda()
# for batch in train_dataloader:
#     model_output = model(batch)
#     break
# model_output
# # # batch
# # # # break
# # # # batch.keys()
# # # # model_output
# # # # # batch = dataloader.dataset[0]
# # # # batch.reference_change_ohlc.times, batch.percentage_change_ohlc.times, batch.reference_date.shape

Creating StockHandlerDataset with offset: 1 and cache: 34700
stocks kept: 502/34700, stocks not found: 0


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


In [3]:
trainer.fit(model=model, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

/home/kalen/ophir/.venv/lib/python3.10/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /home/kalen/ophir/src/ophir/.ophir/model exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


ohlc_predictor.encoder.0._rezero
ohlc_predictor.encoder.1._rezero
ohlc_predictor.encoder.2._rezero
ohlc_predictor.encoder.3._rezero
ohlc_predictor.encoder.4._rezero
ohlc_predictor.encoder.5._rezero
ohlc_predictor.encoder.6._rezero
ohlc_predictor.encoder.7._rezero


/home/kalen/ophir/.venv/lib/python3.10/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type                    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ ohlc_predictor │ OHLCMulitClassPredictor │ 25.5 M │ train │     0 │
└───┴────────────────┴─────────────────────────┴────────┴───────┴───────┘

Trainable params: 25.5 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 25.5 M                                                                                               
Total estimated model params size (MB): 101                                                                        
Modules in train mode: 100                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

/home/kalen/ophir/.venv/lib/python3.10/site-packages/lightning/pytorch/utilities/data.py:79: Trying to infer the 
`batch_size` from an ambiguous collection. The batch size we found is 64. To avoid any miscalculations, use 
`self.log(..., batch_size=batch_size)`.

/home/kalen/ophir/.venv/lib/python3.10/site-packages/lightning/pytorch/utilities/_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/home/kalen/ophir/.venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
